# CAP 6606 Final Project — Structured Entity Extraction with Supervised Fine-Tuning

In this project you will adapt an instruction-tuned causal language model to a structured prediction task using supervised fine-tuning (SFT).

You will fine-tune `HuggingFaceTB/SmolLM2-360M-Instruct` to convert short English passages into a fixed six-field JSON object containing named entities.

## Learning objectives

By the end of the project, you should be able to:

1. format a structured prediction task as an instruction/response example;
2. construct causal-LM training examples containing both context and a target response;
3. explain and implement response-only loss masking;
4. complete the familiar PyTorch forward → loss → backward → update training sequence inside a provided mixed-precision scaffold;
5. compare pretrained and fine-tuned behavior with deterministic evaluation;
6. analyze the remaining errors made by the fine-tuned model.

The primary task metric is **micro-F1 over exact `(category, surface string)` entity mentions**. See `RUBRIC.md` for grading details.


## Connection to Chapter 7

This project intentionally follows the supervised instruction-fine-tuning workflow from Chapter 7. The core idea is unchanged:

```text
Chapter 7                         This project
-----------------------------     -----------------------------
instruction + input               system + user messages
response                          assistant JSON response
InstructionDataset                SFTDataset
custom collate function           provided collate function
forward → loss → backward         forward → loss → backward
optimizer update                  optimizer update
generate responses                generate structured responses
evaluate responses                deterministic micro-F1
```

There are two implementation differences to notice.

**1. Modern chat formatting.** Chapter 7 represents the instruction, input, and response as formatted text. SmolLM2 is already instruction-tuned and provides a native chat template, so we represent the same roles as `system`, `user`, and `assistant` messages.

**2. Response-only supervision.** Chapter 7's main implementation trains on the complete instruction/response sequence. In this project we make one refinement: the instruction remains input context, but only the assistant JSON tokens contribute to the loss. You will implement this small change explicitly.

Everything related to Colab mixed precision, gradient scaling, and numerical-stability checks is **provided infrastructure**. You are responsible for the same core optimization sequence you have already seen in the text.


## Task definition

Every passage must be mapped to exactly these six fields:

```json
{
  "person": [],
  "location": [],
  "group": [],
  "creative_work": [],
  "product": [],
  "medical": []
}
```

Each value is an array of **exact surface strings copied from the input passage**, in order of appearance. If a category is absent, its value is an empty array. Exact boundaries matter.


## 1. Environment setup — provided

Run this notebook on a Google Colab GPU runtime. The reference workflow was tested on a Tesla T4.


In [ ]:
!pip -q install "transformers>=4.46,<5" "accelerate>=1.0,<2"

import hashlib, json, math, random, shutil, subprocess, time
from collections import Counter
from pathlib import Path

import torch
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, get_linear_schedule_with_warmup

SEED = 6606
random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

assert torch.cuda.is_available(), "Select a GPU runtime in Colab before continuing."
DEVICE = torch.device("cuda")
GPU_NAME = torch.cuda.get_device_name(0)
GPU_GB = torch.cuda.get_device_properties(0).total_memory / 2**30
print(f"GPU: {GPU_NAME} ({GPU_GB:.1f} GiB)")


## 2. Load the frozen course data — provided

The notebook clones the public course repository and reads the frozen training and public-development files directly from `data/`.

Do not modify the course data files.


In [ ]:
REPO_URL = "https://github.com/jhs-39/6606_Final_Project.git"
REPO_DIR = Path("/content/6606_Final_Project")
DATA_DIR = REPO_DIR / "data"

if not (DATA_DIR / "train.jsonl").exists():
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    result = subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)],
        text=True,
        capture_output=True,
    )
    if result.returncode != 0:
        raise RuntimeError(f"Repository clone failed:\n{result.stderr}")

EXPECTED_SHA256 = {
    "train.jsonl": "f515ba2db9607470b7a95450b41c0fd02367586a9a7e9204ffcf77e8869a1ac7",
    "dev_public.jsonl": "f9ffd05d57e7019870d7429c6b14b327fba19ee63e086230fecf8ac4c2059e54",
}

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

for name, expected in EXPECTED_SHA256.items():
    assert sha256(DATA_DIR / name) == expected, f"{name}: frozen-data hash mismatch"

def read_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

train_records = read_jsonl(DATA_DIR / "train.jsonl")
dev_records = read_jsonl(DATA_DIR / "dev_public.jsonl")
assert len(train_records) == 6000
assert len(dev_records) == 1000
print(f"Loaded {len(train_records):,} training examples and {len(dev_records):,} public-development examples.")
print(json.dumps(train_records[0], indent=2, ensure_ascii=False))


### Checkpoint question 1

Why are the six target values arrays rather than individual strings? Give one concrete situation in which an array is necessary.

**Your answer:**

> TODO


## 3. Experiment configuration — defaults provided

The settings below are a **tested default configuration** that should run comfortably in the intended Colab environment and provide a strong starting point. You may use them unchanged, but you are also encouraged to experiment with training settings and observe how they affect learning and final performance.

Reasonable settings to explore include the **number of epochs, learning rate, effective batch size** (via `MICRO_BATCH` and `GRAD_ACCUM`), **weight decay**, and **warmup ratio**. You do not need to perform an exhaustive hyperparameter search; a small number of purposeful comparisons is enough to learn from the experiment.

Keep the model, frozen course data, target schema, and deterministic evaluator unchanged so that your results remain comparable. If you change the defaults, record the settings you used and briefly discuss their effect.


In [ ]:
MODEL_NAME = "HuggingFaceTB/SmolLM2-360M-Instruct"
MAX_LENGTH = 256
MAX_NEW_TOKENS = 160
EVAL_BATCH_SIZE = 16

# Tested defaults. You may change the training settings below to run controlled experiments.
EPOCHS = 1
MICRO_BATCH = 4
GRAD_ACCUM = 8
LEARNING_RATE = 5e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.05

OUTPUT_DIR = Path("/content/outputs/smollm2_360m_multiconer_student")

train_work = list(train_records)
dev_work = list(dev_records)
print({
    "epochs": EPOCHS,
    "micro_batch": MICRO_BATCH,
    "grad_accum": GRAD_ACCUM,
    "effective_batch": MICRO_BATCH * GRAD_ACCUM,
    "lr": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "warmup_ratio": WARMUP_RATIO,
})


## 4. TODO 1 — Format the instruction/response example

Chapter 7 builds an instruction, optional input, and response. Here the roles are represented with SmolLM2's chat template:

- **system** → task definition;
- **user** → passage to process;
- **assistant** → gold JSON response.

Complete the three functions below. Do not change the six required fields or normalize the source text.


In [ ]:
KEYS = ("person", "location", "group", "creative_work", "product", "medical")
SYSTEM_PROMPT = (
    "You extract named entities from text. Return only one JSON object with exactly these six keys: "
    "person, location, group, creative_work, product, medical. Each value must be a JSON array of "
    "exact surface strings copied from the input, in order of appearance. Use an empty array when a category is absent."
)

def target_json(record):
    # TODO: serialize record["target"] as compact valid JSON without changing its strings.
    raise NotImplementedError

def user_message(record):
    # TODO: return an extraction instruction followed by the exact source passage.
    raise NotImplementedError

def build_prompt_messages(record):
    # TODO: return the system message followed by the user message.
    raise NotImplementedError


In [ ]:
# Provided checks for TODO 1
messages = build_prompt_messages(train_records[0])
assert isinstance(messages, list) and len(messages) == 2
assert messages[0]["role"] == "system" and messages[0]["content"] == SYSTEM_PROMPT
assert messages[1]["role"] == "user" and train_records[0]["text"] in messages[1]["content"]
serialized = target_json(train_records[0])
parsed = json.loads(serialized)
assert tuple(parsed.keys()) == KEYS
assert parsed == train_records[0]["target"]
print(messages)
print(serialized)
print("Prompt/target checks passed.")


## 5. Load the pretrained model — provided

The checkpoint is already instruction-tuned, but it has not been fine-tuned for this six-field extraction task. Model loading and Colab precision details are provided.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float32).to(DEVICE)
model.config.use_cache = False
model.gradient_checkpointing_enable()
model.enable_input_require_grads()
print(f"Parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")


### Checkpoint question 2

What is the difference between an **instruction-tuned model** and a model that has been **fine-tuned specifically for this extraction task**?

**Your answer:**

> TODO


## 6. TODO 2 — Make the Chapter 7 objective response-only

Chapter 7 trains the causal LM on the complete formatted sequence. Here we keep the prompt as context but exclude it from the supervised loss. Hugging Face causal-LM losses ignore label positions set to `-100`.

Complete the single `labels = ...` line. Prompt positions must be `-100`; assistant-response positions must contain their ordinary token IDs.


In [ ]:
def encode_training_record(record):
    prompt_messages = build_prompt_messages(record)
    full_messages = prompt_messages + [{"role": "assistant", "content": target_json(record)}]

    prompt_ids = tokenizer.apply_chat_template(prompt_messages, tokenize=True, add_generation_prompt=True)
    full_ids = tokenizer.apply_chat_template(full_messages, tokenize=True, add_generation_prompt=False)

    assert full_ids[:len(prompt_ids)] == prompt_ids, "Chat-template prefix assumption failed."
    if len(full_ids) > MAX_LENGTH:
        raise ValueError(f"Sequence length {len(full_ids)} exceeds MAX_LENGTH={MAX_LENGTH}.")

    # TODO: one line. Mask prompt tokens and supervise assistant-response tokens.
    labels = None

    return {"input_ids": full_ids, "attention_mask": [1] * len(full_ids), "labels": labels}


In [ ]:
# Provided checks for TODO 2
example = encode_training_record(train_records[0])
assert example["labels"] is not None
assert len(example["input_ids"]) == len(example["labels"])
first_supervised = next(i for i, x in enumerate(example["labels"]) if x != -100)
assert all(x == -100 for x in example["labels"][:first_supervised])
assert example["labels"][first_supervised:] == example["input_ids"][first_supervised:]
print("Ignored prompt tokens:", sum(x == -100 for x in example["labels"]))
print("Supervised response tokens:", sum(x != -100 for x in example["labels"]))
print("Response-only masking checks passed.")


### Checkpoint question 3

Compare this label construction with the Chapter 7 objective. What contribution to the loss have we removed, and why is that reasonable for this task?

**Your answer:**

> TODO


## 7. Deterministic evaluator — provided; do not modify

Malformed/schema-invalid output is scored as an empty prediction. Micro-F1 is computed globally over exact `(category, surface string)` mentions, with repeated mentions counted by multiplicity.


In [ ]:
EMPTY_TARGET = {k: [] for k in KEYS}

def parse_prediction_with_status(text):
    stripped = text.strip()
    if not stripped:
        return {k: [] for k in KEYS}, "empty_text"
    try:
        obj = json.loads(stripped)
    except Exception:
        return {k: [] for k in KEYS}, "invalid_json"
    if not isinstance(obj, dict):
        return {k: [] for k in KEYS}, "not_object"
    if set(obj.keys()) != set(KEYS) or len(obj) != len(KEYS):
        return {k: [] for k in KEYS}, "wrong_keys"
    for k in KEYS:
        if not isinstance(obj[k], list) or not all(isinstance(x, str) for x in obj[k]):
            return {k: [] for k in KEYS}, "wrong_value_type"
    return {k: obj[k] for k in KEYS}, "valid"

def parse_prediction(text):
    return parse_prediction_with_status(text)[0]

def mention_counter(target):
    c = Counter()
    for category in KEYS:
        for surface in target[category]:
            c[(category, surface)] += 1
    return c

def micro_f1(golds, preds):
    tp = fp = fn = 0
    for gold, pred in zip(golds, preds):
        g, p = mention_counter(gold), mention_counter(pred)
        keys = set(g) | set(p)
        tp += sum(min(g[k], p[k]) for k in keys)
        fp += sum(max(p[k] - g[k], 0) for k in keys)
        fn += sum(max(g[k] - p[k], 0) for k in keys)
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {"micro_f1": f1, "precision": precision, "recall": recall, "tp": tp, "fp": fp, "fn": fn}

def generation_diagnostics(raw):
    counts = Counter(parse_prediction_with_status(x)[1] for x in raw)
    return {"status_counts": dict(counts), "nonempty_raw": sum(bool(x.strip()) for x in raw), "n": len(raw)}


## 8. Generation and evaluation helpers — provided

This plays the same role as Chapter 7's response-generation stage, but our outputs can be scored deterministically.


In [ ]:
@torch.inference_mode()
def predict_records(model, records, batch_size=EVAL_BATCH_SIZE):
    model.eval()
    old_cache = model.config.use_cache
    model.config.use_cache = True
    outputs = []
    try:
        for start in range(0, len(records), batch_size):
            batch = records[start:start + batch_size]
            prompts = []
            for record in batch:
                prompts.append(tokenizer.apply_chat_template(build_prompt_messages(record), tokenize=False, add_generation_prompt=True))
            toks = tokenizer(prompts, return_tensors="pt", padding=True, truncation=False).to(DEVICE)
            with torch.autocast(device_type="cuda", dtype=torch.float16):
                generated = model.generate(**toks, max_new_tokens=MAX_NEW_TOKENS, do_sample=False, pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id)
            prompt_width = toks["input_ids"].shape[1]
            outputs.extend(x.strip() for x in tokenizer.batch_decode(generated[:, prompt_width:], skip_special_tokens=True))
    finally:
        model.config.use_cache = old_cache
    return outputs

def evaluate_model(model, records):
    t0 = time.time()
    raw = predict_records(model, records)
    parsed = [parse_prediction(x) for x in raw]
    metrics = micro_f1([r["target"] for r in records], parsed)
    metrics["seconds"] = time.time() - t0
    return metrics, raw, parsed, generation_diagnostics(raw)


## 9. Pretrained baseline — run this before training

The pretrained model is instruction-tuned, so inspect what it already understands before task-specific SFT.


In [ ]:
baseline_metrics, baseline_raw, baseline_parsed, baseline_diag = evaluate_model(model, dev_work)
print(json.dumps(baseline_metrics, indent=2))
for i in range(3):
    print("\nTEXT:", dev_work[i]["text"])
    print("GOLD:", target_json(dev_work[i]))
    print("RAW :", repr(baseline_raw[i]))


### Checkpoint question 4

From the baseline examples, what does the pretrained instruction-tuned model already appear to understand? What parts of the exact extraction contract does it fail to follow?

**Your answer:**

> TODO


## 10. Dataset and collator — provided

Like Chapter 7's `InstructionDataset` and custom collator, these objects prepare tokenized examples and pad each batch. Padding labels use `-100` so padding does not contribute to the loss.


In [ ]:
class SFTDataset(Dataset):
    def __init__(self, records):
        self.items = [encode_training_record(r) for r in records]
    def __len__(self):
        return len(self.items)
    def __getitem__(self, i):
        return self.items[i]

def collate(batch):
    max_len = max(len(x["input_ids"]) for x in batch)
    input_ids, attention_mask, labels = [], [], []
    for x in batch:
        pad = max_len - len(x["input_ids"])
        input_ids.append(x["input_ids"] + [tokenizer.pad_token_id] * pad)
        attention_mask.append(x["attention_mask"] + [0] * pad)
        labels.append(x["labels"] + [-100] * pad)
    return {
        "input_ids": torch.tensor(input_ids, dtype=torch.long),
        "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
        "labels": torch.tensor(labels, dtype=torch.long),
    }

train_dataset = SFTDataset(train_work)
generator = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(train_dataset, batch_size=MICRO_BATCH, shuffle=True, generator=generator, collate_fn=collate)
lengths = [len(x["input_ids"]) for x in train_dataset.items]
print(f"sequence lengths: min={min(lengths)}, max={max(lengths)}, mean={sum(lengths)/len(lengths):.1f}")


## 11. TODO 3 — Complete the familiar SFT optimization step

The mixed-precision and gradient-accumulation machinery below is provided infrastructure. Focus on the same core sequence used throughout the text:

```text
forward pass → loss → backward pass → optimizer update → clear gradients
```

Fill only the marked TODO lines. Do not remove the provided `autocast`, `GradScaler`, accumulation, clipping, scheduler, or finite-value checks.


In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
updates_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM)
total_updates = updates_per_epoch * EPOCHS
warmup_steps = max(1, int(total_updates * WARMUP_RATIO))
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_updates)
scaler = torch.amp.GradScaler("cuda", enabled=True)

model.train()
optimizer.zero_grad(set_to_none=True)
global_step = 0
train_start = time.time()
loss_sum = 0.0
loss_count = 0

for epoch in range(EPOCHS):
    for batch_idx, batch in enumerate(train_loader, 1):
        batch = {k: v.to(DEVICE) for k, v in batch.items()}

        with torch.autocast(device_type="cuda", dtype=torch.float16):
            # TODO: run the model forward on this batch.
            outputs = None
            # TODO: obtain the model's causal-LM loss.
            raw_loss = None

        assert raw_loss is not None
        if not torch.isfinite(raw_loss):
            raise RuntimeError(f"Non-finite loss at epoch={epoch+1}, batch={batch_idx}")
        loss_sum += float(raw_loss.detach())
        loss_count += 1

        # TODO: backpropagate the loss, accounting for GRAD_ACCUM and using scaler.

        should_step = (batch_idx % GRAD_ACCUM == 0 or batch_idx == len(train_loader))
        if should_step:
            scaler.unscale_(optimizer)
            grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            if not torch.isfinite(grad_norm):
                raise RuntimeError(f"Non-finite gradient norm at update {global_step + 1}")

            # TODO: perform the scaled optimizer update.
            # TODO: update the scaler.
            scheduler.step()
            # TODO: clear gradients for the next accumulation window.
            global_step += 1

            if global_step % 25 == 0 or global_step == 1:
                print(f"epoch={epoch+1} step={global_step}/{total_updates} mean_loss={loss_sum/loss_count:.4f}")

train_seconds = time.time() - train_start
print(f"Training completed in {train_seconds/60:.1f} minutes; mean loss={loss_sum/loss_count:.4f}")


## 12. Post-SFT evaluation — provided

Use exactly the same public-development set and evaluator as the pretrained baseline.


In [ ]:
post_metrics, post_raw, post_parsed, post_diag = evaluate_model(model, dev_work)
print("BASELINE")
print(json.dumps(baseline_metrics, indent=2))
print("\nPOST-SFT")
print(json.dumps(post_metrics, indent=2))
print("\nImprovement in micro-F1:", post_metrics["micro_f1"] - baseline_metrics["micro_f1"])


### Checkpoint question 5

What changed after supervised fine-tuning? Discuss both the structured output format and extraction quality.

**Your answer:**

> TODO


## 13. TODO 4 — Error analysis

Inspect approximately five representative post-SFT errors. For each example, identify one or more useful error categories:

- false negative;
- false positive;
- wrong coarse category;
- entity-boundary error;
- formatting/schema error;
- other clearly explained failure.

For each example, compare the passage, gold target, and model prediction and explain why the prediction is wrong under the exact evaluator. Finish with one short paragraph identifying the most important recurring error pattern you observe.

### Your analysis

> TODO


In [ ]:
# Provided helper: print high-error examples for analysis.
def error_size(gold, pred):
    g, p = mention_counter(gold), mention_counter(pred)
    keys = set(g) | set(p)
    return sum(max(p[k] - g[k], 0) + max(g[k] - p[k], 0) for k in keys)

ranked = sorted(range(len(dev_work)), key=lambda i: error_size(dev_work[i]["target"], post_parsed[i]), reverse=True)
for i in ranked[:10]:
    print("\n" + "=" * 80)
    print("TEXT:", dev_work[i]["text"])
    print("GOLD:", target_json(dev_work[i]))
    print("PRED:", json.dumps(post_parsed[i], ensure_ascii=False))


## 14. Save your fine-tuned checkpoint — provided


In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(OUTPUT_DIR / "model")
tokenizer.save_pretrained(OUTPUT_DIR / "model")
results = {
    "model_name": MODEL_NAME,
    "max_length": MAX_LENGTH,
    "epochs": EPOCHS,
    "micro_batch": MICRO_BATCH,
    "grad_accum": GRAD_ACCUM,
    "learning_rate": LEARNING_RATE,
    "baseline": baseline_metrics,
    "post_sft": post_metrics,
    "gpu": GPU_NAME,
}
with open(OUTPUT_DIR / "student_results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)
print("Saved to", OUTPUT_DIR)


## Submission checklist

Before submitting, verify that:

- [ ] all TODO code sections are complete;
- [ ] all checkpoint questions are answered;
- [ ] the pretrained baseline was evaluated before training;
- [ ] at least one complete SFT training run was performed;
- [ ] the training settings used for your reported result are recorded;
- [ ] baseline and post-SFT micro-F1 are shown;
- [ ] approximately five post-SFT errors are analyzed;
- [ ] the fine-tuned checkpoint and `student_results.json` are saved as directed;
- [ ] the notebook runs in order in the intended Colab environment.

You are welcome to report additional experimental runs if you tried different training settings. See `RUBRIC.md` for grading details.
